In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# تصوير توقيت الـ Circuit

<Accordion>
<AccordionItem title="إصدارات الحزم">

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
على الرغم من أن [رسم الخط الزمني](/guides/visualize-circuit-timing) المضمَّن في Qiskit مفيد للـ Circuits الثابتة، فإنه قد لا يعكس بدقة توقيت [الـ Circuits الديناميكية](/guides/classical-feedforward-and-control-flow) بسبب العمليات الضمنية مثل البث وتحديد الفروع. كجزء من دعم الـ Circuits الديناميكية، يُعيد Qiskit Runtime معلومات توقيت الـ دائرة الدقيقة داخل نتائج المهمة عند الطلب.

> **Note:** - هذه وظيفة تجريبية. في حالة إصدار المعاينة وبالتالي قابلة للتغيير.
> - تنطبق هذه الوظيفة فقط على مهام Qiskit Runtime Sampler.
> - على الرغم من أن إجمالي وقت الـ دائرة يُعاد في بيانات وصفية "compilation"، إلا أن هذا ليس الوقت المُستخدَم للفوترة (وقت QPU).
### تمكين استرداد بيانات التوقيت
لتمكين استرداد بيانات التوقيت، اضبط علامة `scheduler_timing` التجريبية على `True` عند تشغيل مهمة الـ primitive.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### الوصول إلى بيانات توقيت الـ دائرة
عند الطلب، تُعاد بيانات توقيت الـ دائرة لكل PUB في البيانات الوصفية لنتيجة المهمة، ضمن `["compilation"]["scheduler_timing"]["timing"]`. يحتوي هذا الحقل على معلومات التوقيت الخام. لعرض معلومات التوقيت، استخدم أداة التصوير المضمَّنة كما هو موضح في قسم [تصوير التوقيتات](#visualize-timings).

استخدم الكود التالي للوصول إلى بيانات توقيت الـ دائرة للـ PUB الأول:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### فهم بيانات التوقيت الخام
على الرغم من أن تصوير بيانات توقيت الـ دائرة باستخدام طريقة `draw_circuit_schedule_timing` هو الاستخدام الأكثر شيوعًا، قد يكون من المفيد فهم هيكل بيانات التوقيت الخام المُعادة. يمكن أن يساعدك ذلك مثلاً في استخراج المعلومات برمجيًا.

بيانات التوقيت المُعادة في `["compilation"]["scheduler_timing"]["timing"]` هي قائمة من السلاسل النصية. كل سلسلة تمثل تعليمة واحدة على قناة ما، وهي مفصولة بفاصلة إلى أنواع البيانات التالية:

- `Branch` - يُحدِّد ما إذا كانت التعليمة في تدفق تحكم (then / else) أو فرع رئيسي.
- `Instruction` - البوابة والـ qubit المُراد التشغيل عليه.
- `Channel` - القناة المُخصَّصة للتعليمة. يمكن أن تكون إحدى التالية:
      - `qubit x` - قناة الإشارة للـ qubit _x_.
      - `AWGRx_y` (مولِّد موجة عشوائية لقراءة المخرج) - تُستخدَم بواسطة قنوات القراءة للتواصل عند قياس الـ qubits. تتوافق الوسيطتان _x_ و_y_ مع معرِّف أداة القراءة ورقم الـ qubit على التوالي.
- `T0` - وقت بدء التعليمة ضمن الجدول الكامل
- `Duration` - مدة التعليمة بوحدات ثوانٍ _dt_، حيث 1 dt = دورة جدولة واحدة. يمكنك إيجاد قيمة `dt` للـ Backend باستخدام [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - نوع عملية النبضة المُستخدَمة.

مثال:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### تصوير التوقيتات
مع `qiskit-ibm-runtime` الإصدار v0.43.0 أو أحدث، يمكنك تصوير توقيتات الـ دائرة. للتصوير، تحتاج أولاً إلى تحويل البيانات الوصفية للنتيجة إلى `fig` باستخدام [طريقة `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). تُعيد هذه الطريقة شكل `plotly`، يمكنك عرضه مباشرةً أو حفظه في ملف أو كليهما. لمزيد من المعلومات حول أوامر `plotly` المُستخدَمة، راجع [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) و[`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Hovering over the output shows information such as the start, finish, and duration.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Example of a generated figure')
#### فهم الشكل المولَّد
تُعبِّر صورة بيانات توقيت الـ دائرة المُخرَجة بواسطة `draw_circuit_schedule_timing` عن المعلومات التالية:

- المحور السيني هو الوقت بوحدات ثوانٍ _dt_، حيث 1 dt = دورة جدولة واحدة. يمكنك إيجاد قيمة `dt` للـ Backend باستخدام [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- المحور الصادي هو القناة (فكِّر في القنوات كأدوات تُصدِر نبضات).
    - `Receive channel` - القناة الوحيدة التي ليست أداة بذاتها. هي تعليمة تُشغَّل على جميع القنوات التي تكون جزءًا من إجراء تواصل مع المحور في ذلك الوقت.
    - `qubit x` - قناة الإشارة للـ qubit x.
    - `AWGRx_y` (مولِّد موجة عشوائية لقراءة المخرج) - تُستخدَم بواسطة قنوات القراءة للتواصل عند قياس الـ qubits. تتوافق الوسيطتان _x_ و_y_ مع معرِّف أداة القراءة ورقم الـ qubit على التوالي.
    - `Hub` - يتحكم في البث.

بالإضافة إلى ذلك، كل تعليمة لها تنسيق *X_Y*، حيث *X* هو اسم التعليمة و*Y* هو نوع النبضة. نوع `play` يُطبِّق نبضات التحكم، ونوع `capture` يُسجِّل حالة الـ qubit. يمكنك أيضًا التمرير فوق كل تعليمة للحصول على مزيد من التفاصيل. على سبيل المثال، يُظهِر الشكل السابق نبضة تحكم لبوابة X المُطبَّقة على الـ qubit 10 عند 1161 dt.
### مثال شامل
يُوضِّح هذا المثال كيفية تمكين الخيار، والحصول على معلومات توقيت الـ دائرة من البيانات الوصفية، وعرضها كصورة.

أولاً، هيِّئ البيئة وحدِّد الـ Circuits وحوِّلها إلى Circuits من نوع ISA، وحدِّد المهام وشغِّلها.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

بعد ذلك، احصل على توقيت جدول الـ دائرة: